# Deconvolution with a measured PSF

Richardson-Lucy deconvolution (`calmutils.restoration.rl_deconvolve`) of all
TIFFs in a dataset folder with a PSF measured by `psf_extraction.ipynb`.

- Inputs: `in_path/<tif_subdir>/*.tif` (ImageJ TIFFs with pixel size, or set `psz`)
- PSF: `psf_path` pointing to a PSF tiff from the extraction notebook (e.g.
  `ALL_psf_distilled.tif`); its voxel size is read from the `.json` sidecar
  written next to it (or set `psf_psz`). The PSF is band-limited-resampled onto
  each image's grid (`resample_psf`), so PSF and image pixel sizes may differ.
- Outputs in `in_path/<out_subdir>/`: `<stem>_deconv.tif` + JSON sidecar per
  image, `report.png`, `recipe_parameters.json`.

The RL core uses `edge="reflect"` padding (no wrap ghosts at image borders),
optional per-iteration OTF band limit (`bandlimit`, in cycles per nm) and an
optional denominator floor (`damp`). Output is flux-preserving (not
renormalized). No TV regularization - see `rl_deconvolve` docstring.

Run interactively, or via papermill / `recipe_runner.py`
(`done_if_exists: "{in_path}/deconv"`).

In [ ]:
# dataset folder with the images to deconvolve
in_path = '/data/sample_dataset'

# subfolder with tiffs (relative to in_path), '' if directly in in_path
tif_subdir = 'tif'

# results subfolder (relative to in_path)
out_subdir = 'deconv'

# PSF from psf_extraction.ipynb, e.g. .../psf/ALL_psf_distilled.tif
# (its voxel size is read from the .json sidecar next to it)
psf_path = '/data/beads_dataset/psf/ALL_psf_distilled.tif'
psf_psz = None    # [z, y, x] in nm, override if there is no sidecar

# image voxel size [z, y, x] in nm; None = read from ImageJ tiff metadata
psz = None

# RL parameters
num_iter = 25
# per-axis OTF cutoff in cycles per nm (scalar or [z, y, x]); None = off
bandlimit = None
# denominator floor relative to the image, e.g. 1e-3; None = off
damp = None
edge = 'reflect'    # 'reflect' | 'periodic'

# background: subtract this percentile of each image before RL; None = no subtraction
background_percentile = 1.0

In [ ]:
# << auto-managed by scripts/inject_parameter_logging.py - do not edit >>
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'recipe_params.py').is_file():
        sys.path.insert(0, str(_p))
        break
import recipe_params
recipe_params.capture(globals(), notebook='restoration/deconvolution.ipynb')

In [ ]:
import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from matplotlib import pyplot as plt

from calmutils.imageio.tiff_imagej import save_tiff_imagej, get_imagej_tiff_pixel_size
from calmutils.restoration import rl_deconvolve, resample_psf, psf_metrics

in_path = Path(in_path)
tif_dir = in_path / tif_subdir if tif_subdir else in_path
out_path = in_path / out_subdir
out_path.mkdir(parents=True, exist_ok=True)
psf_path = Path(psf_path)
print(f'input images : {tif_dir}')
print(f'psf          : {psf_path}')
print(f'output       : {out_path}')

In [ ]:
# --- load PSF (voxel size from its sidecar, unless psf_psz overrides) ---
psf = tifffile.imread(psf_path).astype(float)
if psf.ndim != 3:
    raise ValueError(f'PSF must be a 3D (z, y, x) array, got {psf.shape}')
if psf_psz is None:
    sidecar = Path(str(psf_path) + '.json')
    if not sidecar.is_file():
        raise FileNotFoundError(f'no sidecar {sidecar} - use a PSF written by '
                                'psf_extraction.ipynb, or set psf_psz')
    psf_psz = np.asarray(json.load(open(sidecar))['psz_nm'], dtype=float)
else:
    psf_psz = np.asarray(psf_psz, dtype=float)
print(f'psf shape {psf.shape}, psz_nm {psf_psz}, sum {psf.sum():.4f}')
print('psf metrics:', {k: (np.round(v, 3) if isinstance(v, tuple) else round(v, 5))
                       for k, v in psf_metrics(psf, psf_psz).items()
                       if k in ('fwhm_phys', 'sigma_px', 'peak_offset_px',
                                'sidelobe_ratio', 'edge_level')})

# --- input images + per-image pixel size ---
tif_files = sorted(set(tif_dir.glob('*.tif')) | set(tif_dir.glob('*.tiff')))
if not tif_files:
    raise FileNotFoundError(f'no .tif/.tiff files in {tif_dir}')

def image_psz(f):
    if psz is not None:
        return np.asarray(psz, dtype=float)
    s, unit = get_imagej_tiff_pixel_size(f)
    to_nm = {'nm': 1.0, 'um': 1e3, 'µm': 1e3, 'mm': 1e6, 'cm': 1e7, 'm': 1e9}.get(unit)
    if to_nm is None:
        raise ValueError(f'{f.name}: cannot interpret pixel unit {unit!r} - set psz')
    return np.asarray(s, dtype=float) * to_nm

print(f'{len(tif_files)} image(s): {[f.name for f in tif_files]}')

In [ ]:
# --- deconvolve each image ---
summary = []
for f in tif_files:
    img = tifffile.imread(f).astype(float)
    assert img.ndim == 3, f'{f.name}: expected a 3D stack (z, y, x), got {img.shape}'
    img_psz = image_psz(f)

    if background_percentile is not None:
        img = img - np.percentile(img, background_percentile)

    # PSF onto the image grid (band-limited, centroid stays on center voxel)
    psf_r, psz_eff = resample_psf(psf, psf_psz, img_psz)
    if np.any(np.abs(psz_eff / img_psz - 1) > 0.05):
        warnings.warn(f'{f.name}: effective PSF voxel size {psz_eff} deviates '
                      f'>5% from image grid {img_psz}')

    bl = None
    if bandlimit is not None:
        bl = np.broadcast_to(np.asarray(bandlimit, dtype=float), (3,)) * img_psz  # -> cycles/voxel

    dec = rl_deconvolve(img, psf_r, num_iter=num_iter, bandlimit=bl,
                        edge=edge, damp=damp)

    out_tif = out_path / f'{f.stem}_deconv.tif'
    save_tiff_imagej(str(out_tif), dec.astype(np.float32), axes='ZYX',
                     pixel_size=list(img_psz), distance_unit='nm')
    sidecar = {
        'source_image': f.name, 'psf_path': str(psf_path), 'psf_psz_nm': psf_psz.tolist(),
        'image_psz_nm': img_psz.tolist(), 'psf_shape_on_image_grid': list(psf_r.shape),
        'psf_psz_eff_nm': psz_eff.tolist(),
        'num_iter': num_iter, 'bandlimit_cycles_per_nm': bandlimit, 'damp': damp,
        'edge': edge, 'background_percentile': background_percentile,
        'flux_in': float(img.sum()), 'flux_out': float(dec.sum()),
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    with open(str(out_tif) + '.json', 'w') as fh:
        json.dump(sidecar, fh, indent=2)
    summary.append({'file': f.name, 'flux_ratio': float(dec.sum() / max(img.sum(), 1e-12))})
    print(f'{f.name}: psf {psf.shape}->{psf_r.shape} on grid, deconvolved '
          f'({num_iter} it) -> {out_tif.name}, flux out/in '
          f'{summary[-1]["flux_ratio"]:.3f}')

pd.DataFrame(summary).to_csv(out_path / 'deconv_summary.csv', index=False)

In [ ]:
# --- quick before/after report for the first image ---
f = tif_files[0]
img = tifffile.imread(f).astype(float)
if background_percentile is not None:
    img = img - np.percentile(img, background_percentile)
dec = tifffile.imread(out_path / f'{f.stem}_deconv.tif').astype(float)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img.max(axis=0), cmap='gray')
axs[0].set_title(f'{f.name} input (max-proj)')
axs[1].imshow(dec.max(axis=0), cmap='gray')
axs[1].set_title(f'deconvolved, {num_iter} RL iters')
fig.tight_layout()
fig.savefig(out_path / 'report.png', dpi=150)
plt.show()
print('saved to', out_path)
for p in sorted(out_path.iterdir()):
    print('  ', p.name)